# AI Log



In [26]:
# %% [code]
import requests
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

print("✅ Imports for Elhub klar")


✅ Imports for Elhub klar


In [27]:

# %% [code]
import os
import sys
import subprocess

# Stopp gammel SparkSession hvis den finnes
try:
    spark.stop()
except:
    pass

# --- Sett opp JAVA_HOME (prøv å finne Java 11 automatisk på macOS) ---
try:
    java_home = subprocess.check_output(
        ["/usr/libexec/java_home", "-v", "11"]
    ).decode().strip()
    os.environ["JAVA_HOME"] = java_home
    print("JAVA_HOME satt til:", java_home)
except Exception as e:
    print("⚠️ Klarte ikke finne Java 11 med /usr/libexec/java_home.")
    print("   Sjekk at du har Java 11 installert. Feil:", e)

# Sørg for at PySpark bruker samme Python som Jupyter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

# --- SparkSession med Cassandra + riktig driver-adresse ---
spark = (
    SparkSession.builder
    .appName("Elhub Production & Consumption 2021-2024")
    .master("local[*]")  # lokal kjøring
    # 👇 Disse to er viktige for feilen du ser
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    # Cassandra-connector
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"
    )
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.cassandra.auth.username", "cassandra")
    .config("spark.cassandra.auth.password", "cassandra")
    .config("spark.cassandra.output.consistency.level", "LOCAL_QUORUM")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .getOrCreate()
)

print("✅ SparkSession koblet til Cassandra (driver på 127.0.0.1)")


# --- MongoDB-klient via secrets.toml ---
import tomllib  # innebygd i Python 3.11+
import certifi
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

# Les .streamlit/secrets.toml
with open("/Users/a.h.sheikh/Desktop/IND320_Git_Job/IND320_1_Project/.streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)

MONGODB_URI = secrets["MONGODB_URI"]  # samme som du viste meg

ca = certifi.where()
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"), tlsCAFile=ca)

# Test tilkobling
try:
    client.admin.command("ping")
    print("✅ MongoDB-tilkobling OK")
except Exception as e:
    print("❌ MongoDB-tilkobling feilet:", e)

# Velg database og collections (samme som før, eller endre navn hvis du vil)
mongo_db = client["example"]
mongo_collection_production = mongo_db["production"]    # produksjon 2021–2024
mongo_collection_consumption = mongo_db["consumption"]  # forbruk 2021–2024
# ADVARSEL: dette sletter HELE databasen "example"
client.drop_database("example")
print("🧹 Droppet database 'example'")

# %% [code]
# --------------------------------------------------
# Elhub-API: felles oppsett og hjelpefunksjoner
# --------------------------------------------------
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"

# Hent prisområder dynamisk (fallback NO1–NO5 hvis noe feiler)
areas_response = requests.get(BASE_URL)
if areas_response.status_code == 200:
    areas = areas_response.json().get("priceAreas", ["NO1", "NO2", "NO3", "NO4", "NO5"])
else:
    print("⚠️ Kunne ikke hente priceAreas fra API. Bruker NO1–NO5 som default.")
    areas = ["NO1", "NO2", "NO3", "NO4", "NO5"]

print("📍 Prisområder:", areas)


def fetch_elhub_window(dataset: str,
                       record_key: str,
                       area: str,
                       start_dt: datetime,
                       end_dt: datetime) -> pd.DataFrame:
    """
    Henter én tidswindow for ett område og ett dataset.
    dataset: f.eks. 'PRODUCTION_PER_GROUP_MBA_HOUR'
    record_key: f.eks. 'productionPerGroupMbaHour' / 'consumptionPerGroupMbaHour'
    """
    api_url = f"{BASE_URL}/{area}"
    params = {
        "dataset": dataset,
        "startDate": start_dt.isoformat(),  # requests URL-enkoder + til %2B
        "endDate": end_dt.isoformat()
    }

    response = requests.get(api_url, params=params)

    if response.status_code != 200:
        print(
            f"⚠️ Feil {response.status_code} for {dataset}, {area}: "
            f"{start_dt.isoformat()} → {end_dt.isoformat()}"
        )
        return pd.DataFrame()

    try:
        data = response.json()
        records = data["data"][0]["attributes"][record_key]
    except (KeyError, IndexError):
        print(
            f"⚠️ Uventet JSON-struktur for {dataset}, {area}: "
            f"{start_dt.isoformat()} → {end_dt.isoformat()}"
        )
        return pd.DataFrame()

    if not records:
        print(
            f"⚠️ Ingen rader for {dataset}, {area}: "
            f"{start_dt.isoformat()} → {end_dt.isoformat()}"
        )
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df["priceArea"] = area
    return df


def fetch_dataset_years(dataset: str,
                        record_key: str,
                        years,
                        areas) -> pd.DataFrame:
    """
    Henter data for flere år og alle prisområder, med månedlige vinduer
    (for å holde oss innenfor maks dato-range til APIet).
    """
    all_dfs = []

    for year in years:
        year_start = datetime.fromisoformat(f"{year}-01-01T00:00:00+02:00")
        year_end = datetime.fromisoformat(f"{year}-12-31T23:59:59+02:00")
        print(f"\n📆 Henter {dataset} for år {year}")

        for area in areas:
            current_start = year_start
            while current_start < year_end:
                current_end = current_start + relativedelta(months=1)
                if current_end > year_end:
                    current_end = year_end

                print(
                    f"   → {dataset} {area}: "
                    f"{current_start.isoformat()} → {current_end.isoformat()}"
                )

                df = fetch_elhub_window(dataset, record_key, area, current_start, current_end)
                if not df.empty:
                    all_dfs.append(df)

                current_start = current_end

    if not all_dfs:
        print("⚠️ Ingen data hentet.")
        return pd.DataFrame()

    final_df = pd.concat(all_dfs, ignore_index=True)
    return final_df

# %% [markdown]
# ## 1️⃣ Produksjon 2022–2024 (PRODUCTION_PER_GROUP_MBA_HOUR)
# Vi forutsetter at produksjons-data for 2021 allerede ligger i Cassandra-tabell `elhub.production_2021`.
# Her henter vi 2022–2024 og **append'er** i samme tabell og i Mongo.

# %% [code]
# Hent produksjonsdata 2022–2024
production_years = range(2022, 2025)  # 2022, 2023, 2024

production_df_22_24 = fetch_dataset_years(
    dataset="PRODUCTION_PER_GROUP_MBA_HOUR",
    record_key="productionPerGroupMbaHour",
    years=production_years,
    areas=areas
)

print("\n✅ Antall rader produksjon 2022–2024:", len(production_df_22_24))

# Lagre til CSV for sikkerhetskopi / sjekk
production_df_22_24.to_csv("elhub_production_2022_2024_all_areas.csv", index=False)
print("💾 Lagret til elhub_production_2022_2024_all_areas.csv")

# %% [code]
# Skriv produksjon 2022–2024 til Cassandra (append til eksisterende tabel med 2021-data)


# %% [code]
# Skriv produksjon 2022–2024 til Cassandra (append til eksisterende tabell med 2021-data)

# Velg kun relevante kolonner til Cassandra
prod_columns = [
    "endTime",
    "lastUpdatedTime",
    "priceArea",
    "productionGroup",
    "quantityKwh",
    "startTime",
]
prod_for_spark = production_df_22_24[prod_columns].copy()

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

prod_schema = StructType([
    StructField("endTime", StringType(), True),
    StructField("lastUpdatedTime", StringType(), True),
    StructField("priceArea", StringType(), True),
    StructField("productionGroup", StringType(), True),
    StructField("quantityKwh", DoubleType(), True),
    StructField("startTime", StringType(), True),
])

spark_prod_df = spark.createDataFrame(prod_for_spark, schema=prod_schema)

# Først: samme casing som vi vil ha i Cassandra
spark_prod_df = (
    spark_prod_df
    .withColumnRenamed("endTime", "endtime")
    .withColumnRenamed("lastUpdatedTime", "lastupdatedtime")
    .withColumnRenamed("priceArea", "pricearea")
    .withColumnRenamed("productionGroup", "productiongroup")
    .withColumnRenamed("quantityKwh", "quantitykwh")
    .withColumnRenamed("startTime", "starttime")
)

# 👉 VIKTIG: velg kun kolonnene som faktisk finnes i tabellen production_2021
spark_prod_df = spark_prod_df.select(
    "pricearea",
    "productiongroup",
    "starttime",
    "quantitykwh"
)

print("Schema som sendes til Cassandra:")
spark_prod_df.printSchema()

(
    spark_prod_df
    .write
    .format("org.apache.spark.sql.cassandra")
    .mode("append")
    .options(keyspace="elhub", table="production_2021")
    .save()
)

print("✅ Produksjon 2022–2024 skrevet til Cassandra (elhub.production_2021)")




# %% [code]
# Skriv produksjon 2022–2024 til MongoDB (append etter 2021-data)

# Her tar vi bare med noen viktige kolonner
prod_for_mongo = production_df_22_24[[
    "priceArea",
    "productionGroup",
    "startTime",
    "quantityKwh"
]].copy()

prod_for_mongo.rename(
    columns={
        "priceArea": "pricearea",
        "productionGroup": "productiongroup",
        "startTime": "starttime",
        "quantityKwh": "quantitykwh",
    },
    inplace=True,
)

prod_docs = prod_for_mongo.to_dict("records")

try:
    if prod_docs:
        mongo_collection_production.insert_many(prod_docs)
        print(f"✅ {len(prod_docs)} produksjonsrader 2022–2024 skrevet til MongoDB")
    else:
        print("⚠️ Ingen produksjonsdokumenter å skrive til MongoDB")
except Exception as e:
    print("⚠️ Klarte ikke å skrive produksjon til MongoDB:", e)

# %% [markdown]
# ## 2️⃣ Forbruk 2021–2024 (CONSUMPTION_PER_GROUP_MBA_HOUR)
# Her bruker vi **nye tabeller** i Cassandra og Mongo:
# - Cassandra: `elhub.consumption_2021_2024`
# - Mongo: collection `consumption` i databasen `example`
#
# ❗ Du må sørge for at Cassandra-tabellen finnes, f.eks. noe sånt i cqlsh (tilpass etter behov):
#
# ```sql
# CREATE TABLE elhub.consumption_2021_2024 (
#     pricearea text,
#     consumptiongroup text,
#     starttime text,
#     endtime text,
#     lastupdatedtime text,
#     quantitykwh double,
#     PRIMARY KEY ((pricearea, consumptiongroup), starttime)
# );
# ```

# %% [code]
# Hent forbruksdata 2021–2024
consumption_years = range(2021, 2025)  # 2021, 2022, 2023, 2024

consumption_df_21_24 = fetch_dataset_years(
    dataset="CONSUMPTION_PER_GROUP_MBA_HOUR",
    record_key="consumptionPerGroupMbaHour",
    years=consumption_years,
    areas=areas
)

print("\n✅ Antall rader forbruk 2021–2024:", len(consumption_df_21_24))

# Lagre til CSV for sjekk
consumption_df_21_24.to_csv("elhub_consumption_2021_2024_all_areas.csv", index=False)
print("💾 Lagret til elhub_consumption_2021_2024_all_areas.csv")

# %% [code]
# Skriv forbruk 2021–2024 til Cassandra (NY tabell)

cons_columns = [
    "endTime",
    "lastUpdatedTime",
    "priceArea",
    "consumptionGroup",
    "quantityKwh",
    "startTime",
]
cons_for_spark = consumption_df_21_24[cons_columns].copy()

cons_schema = StructType([
    StructField("endTime", StringType(), True),
    StructField("lastUpdatedTime", StringType(), True),
    StructField("priceArea", StringType(), True),
    StructField("consumptionGroup", StringType(), True),
    StructField("quantityKwh", DoubleType(), True),
    StructField("startTime", StringType(), True),
])

spark_cons_df = spark.createDataFrame(cons_for_spark, schema=cons_schema)

spark_cons_df = (
    spark_cons_df
    .withColumnRenamed("endTime", "endtime")
    .withColumnRenamed("lastUpdatedTime", "lastupdatedtime")
    .withColumnRenamed("priceArea", "pricearea")
    .withColumnRenamed("consumptionGroup", "consumptiongroup")
    .withColumnRenamed("quantityKwh", "quantitykwh")
    .withColumnRenamed("startTime", "starttime")
)

(
    spark_cons_df
    .write
    .format("org.apache.spark.sql.cassandra")
    .mode("append")
    .options(keyspace="elhub", table="consumption_2021_2024")  # NY tabell
    .save()
)

print("✅ Forbruk 2021–2024 skrevet til Cassandra (elhub.consumption_2021_2024)")

# %% [code]
# Skriv forbruk 2021–2024 til MongoDB (NY collection `consumption`)

cons_for_mongo = consumption_df_21_24[[
    "priceArea",
    "consumptionGroup",
    "startTime",
    "quantityKwh"
]].copy()

cons_for_mongo.rename(
    columns={
        "priceArea": "pricearea",
        "consumptionGroup": "consumptiongroup",
        "startTime": "starttime",
        "quantityKwh": "quantitykwh",
    },
    inplace=True,
)

cons_docs = cons_for_mongo.to_dict("records")

try:
    if cons_docs:
        mongo_collection_consumption.insert_many(cons_docs)
        print(f"✅ {len(cons_docs)} forbruksrader 2021–2024 skrevet til MongoDB")
    else:
        print("⚠️ Ingen forbruksdokumenter å skrive til MongoDB")
except Exception as e:
    print("⚠️ Klarte ikke å skrive forbruk til MongoDB:", e)


JAVA_HOME satt til: /Library/Java/JavaVirtualMachines/zulu-11.jdk/Contents/Home
✅ SparkSession koblet til Cassandra (driver på 127.0.0.1)
✅ MongoDB-tilkobling OK
🧹 Droppet database 'example'
📍 Prisområder: ['NO1', 'NO2', 'NO3', 'NO4', 'NO5']

📆 Henter PRODUCTION_PER_GROUP_MBA_HOUR for år 2022
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-01-01T00:00:00+02:00 → 2022-02-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-02-01T00:00:00+02:00 → 2022-03-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-03-01T00:00:00+02:00 → 2022-04-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-04-01T00:00:00+02:00 → 2022-05-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-05-01T00:00:00+02:00 → 2022-06-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-06-01T00:00:00+02:00 → 2022-07-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-07-01T00:00:00+02:00 → 2022-08-01T00:00:00+02:00
   → PRODUCTION_PER_GROUP_MBA_HOUR NO1: 2022-08-

25/11/22 13:52:19 WARN TaskSetManager: Stage 0 contains a task of very large size (8510 KiB). The maximum recommended task size is 1000 KiB.


✅ Produksjon 2022–2024 skrevet til Cassandra (elhub.production_2021)
✅ 657600 produksjonsrader 2022–2024 skrevet til MongoDB

📆 Henter CONSUMPTION_PER_GROUP_MBA_HOUR for år 2021
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-01-01T00:00:00+02:00 → 2021-02-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-02-01T00:00:00+02:00 → 2021-03-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-03-01T00:00:00+02:00 → 2021-04-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-04-01T00:00:00+02:00 → 2021-05-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-05-01T00:00:00+02:00 → 2021-06-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-06-01T00:00:00+02:00 → 2021-07-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-07-01T00:00:00+02:00 → 2021-08-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-08-01T00:00:00+02:00 → 2021-09-01T00:00:00+02:00
   → CONSUMPTION_PER_GROUP_MBA_HOUR NO1: 2021-09-01T00:00:00+0

25/11/22 13:56:08 WARN TaskSetManager: Stage 1 contains a task of very large size (11637 KiB). The maximum recommended task size is 1000 KiB.


✅ Forbruk 2021–2024 skrevet til Cassandra (elhub.consumption_2021_2024)
✅ 876600 forbruksrader 2021–2024 skrevet til MongoDB


### Datainnsamling og lagring

For både produksjon og forbruk hentes timeverdier for alle prisområder fra Elhub og lagres i **Cassandra**.

- Produksjonsdata 2022–2024 lagres også i **MongoDB**  
- For forbruksdata 2021–2024 er koden for innsending til Mongo implementert tilsvarende, men ved kjøring stopper MongoDB Atlas med en space quota‑feil (`517 MB of 512 MB`)  
- Dette er en begrensning i gratis‑kontoen, ikke i løsningen  
- Fullstendig lagring av forbruk gjøres derfor kun i **Cassandra**

### Oversiktstabell

| Data                  | Cassandra | MongoDB              |
|-----------------------|-----------|----------------------|
| Produksjon 2022–2024  | Ja        | Ja                   |
| Forbruk 2021–2024     | Ja        | Nei (quota‑feil)     |


In [28]:
# %% [code]
from pyspark.sql.functions import substring, col, countDistinct

prod_cass = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="elhub", table="production_2021")
    .load()
)

print("Antall rader totalt i elhub.production_2021:", prod_cass.count())

# Ta en titt på noen rader
prod_cass.select("pricearea", "productiongroup", "starttime", "quantitykwh").show(10, truncate=False)

# Grov sjekk på år per area
prod_with_year = prod_cass.withColumn("year", substring(col("starttime"), 1, 4))

prod_with_year.groupBy("year", "pricearea").count().orderBy("year", "pricearea").show(50)


Antall rader totalt i elhub.production_2021: 657601
+---------+---------------+-------------------+-----------+
|pricearea|productiongroup|starttime          |quantitykwh|
+---------+---------------+-------------------+-----------+
|NO4      |hydro          |2022-01-01 00:00:00|3840983.2  |
|NO4      |hydro          |2022-01-01 01:00:00|3816076.8  |
|NO4      |hydro          |2022-01-01 02:00:00|3827141.0  |
|NO4      |hydro          |2022-01-01 03:00:00|3813465.8  |
|NO4      |hydro          |2022-01-01 04:00:00|3763152.5  |
|NO4      |hydro          |2022-01-01 05:00:00|3766277.5  |
|NO4      |hydro          |2022-01-01 06:00:00|3743969.8  |
|NO4      |hydro          |2022-01-01 07:00:00|3802487.5  |
|NO4      |hydro          |2022-01-01 08:00:00|3913795.2  |
|NO4      |hydro          |2022-01-01 09:00:00|3969193.2  |
+---------+---------------+-------------------+-----------+
only showing top 10 rows



+----+---------+-----+
|year|pricearea|count|
+----+---------+-----+
|2021|      NO1|    1|
|2022|      NO1|43800|
|2022|      NO2|43800|
|2022|      NO3|43800|
|2022|      NO4|43800|
|2022|      NO5|43800|
|2023|      NO1|43800|
|2023|      NO2|43800|
|2023|      NO3|43800|
|2023|      NO4|43800|
|2023|      NO5|43800|
|2024|      NO1|43920|
|2024|      NO2|43920|
|2024|      NO3|43920|
|2024|      NO4|43920|
|2024|      NO5|43920|
+----+---------+-----+



In [29]:
# %% [code]
# Forutsetter at production_df_22_24 fortsatt finnes i minnet
prod_for_mongo = production_df_22_24[[
    "priceArea",
    "productionGroup",
    "startTime",
    "quantityKwh"
]].copy()

prod_for_mongo.rename(
    columns={
        "priceArea": "pricearea",
        "productionGroup": "productiongroup",
        "startTime": "starttime",
        "quantityKwh": "quantitykwh",
    },
    inplace=True,
)

prod_docs = prod_for_mongo.to_dict("records")

if prod_docs:
    mongo_collection_production.insert_many(prod_docs)
    print(f"✅ {len(prod_docs)} produksjonsdokumenter 2022–2024 skrevet til MongoDB")
else:
    print("⚠️ Ingen produksjonsdokumenter å skrive til MongoDB")


✅ 657600 produksjonsdokumenter 2022–2024 skrevet til MongoDB


In [30]:
print("Dokumenter i Mongo 'production':", mongo_collection_production.count_documents({}))
mongo_collection_production.find_one()


Dokumenter i Mongo 'production': 1315200


{'_id': ObjectId('6921b208315343d2396a4e9b'),
 'pricearea': 'NO1',
 'productiongroup': 'hydro',
 'starttime': '2022-01-01T00:00:00+01:00',
 'quantitykwh': 1291422.4}

In [31]:
from pyspark.sql import functions as F
from pymongo import MongoClient
import pandas as pd

# ---- 1) Les CSV med consumption-data ----
csv_path = "/Users/a.h.sheikh/Desktop/IND320_Git_Job/IND320_1_Project/Ass4_Rapporter/elhub_consumption_2021_2024_all_areas.csv"

cons_csv = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(csv_path)
)

# ---- 2) Rydd kolonner og gjør startTime til timestamp ----
cons_clean = cons_csv.select(
    F.col("priceArea").alias("priceArea"),
    F.to_timestamp("startTime").alias("startTime"),
    F.col("consumptionGroup").alias("consumptionGroup"),
    F.col("quantityKwh").alias("quantityKwh")
)

# ---- 3) Begrens til årene 2021–2024 ----
cons_2021_2024 = cons_clean.filter(
    (F.year("startTime") >= 2021) & (F.year("startTime") <= 2024)
)

print("Schema for consumption-data som skal skrives til Mongo:")
cons_2021_2024.printSchema()
print("Eksempelrader:")
cons_2021_2024.show(5)

# ---- 4) Konverter til pandas og skriv til MongoDB med pymongo ----

# NB: Dette laster alt til minnet. Hvis det blir for tungt, må vi ta det i chunks,
# men prøv dette først.
pdf = cons_2021_2024.toPandas()

# Sørg for at startTime er vanlig datetime (pandas → Python datetime)
pdf["startTime"] = pd.to_datetime(pdf["startTime"])

# Koble til Mongo
client = MongoClient("mongodb://localhost:27017")
db = client["elhub"]
coll = db["consumption_2021_2024_by_hour"]

# Tøm eksisterende collection (så vi ikke får duplikater)
coll.drop()

records = pdf.to_dict("records")
if records:
    coll.insert_many(records)

print(f"✅ Skrevet {len(records)} dokumenter til MongoDB: database='elhub', collection='consumption_2021_2024_by_hour'")


Schema for consumption-data som skal skrives til Mongo:
root
 |-- priceArea: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- consumptionGroup: string (nullable = true)
 |-- quantityKwh: double (nullable = true)

Eksempelrader:
+---------+-------------------+----------------+-----------+
|priceArea|          startTime|consumptionGroup|quantityKwh|
+---------+-------------------+----------------+-----------+
|      NO1|2021-01-01 00:00:00|           cabin|  177071.56|
|      NO1|2021-01-01 01:00:00|           cabin|  171335.12|
|      NO1|2021-01-01 02:00:00|           cabin|  164912.02|
|      NO1|2021-01-01 03:00:00|           cabin|  160265.77|
|      NO1|2021-01-01 04:00:00|           cabin|  159828.69|
+---------+-------------------+----------------+-----------+
only showing top 5 rows



✅ Skrevet 876600 dokumenter til MongoDB: database='elhub', collection='consumption_2021_2024_by_hour'
